In [ ]:
!pip install lightgbm xgboost optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 30.6 MB/s eta 0:00:00


In [ ]:
!pip install kaggle

from google.colab import files
files.upload()  # حملي kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c ieee-fraud-detection
!unzip ieee-fraud-detection.zip

In [ ]:
!unzip "ieee-fraud-detection (1).zip"

In [ ]:
!pip install catboost imbalanced-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 27.5 MB/s eta 0:00:00


In [ ]:
TRAIN_TRANSACTION = "/content/train_transaction.csv"
TRAIN_IDENTITY    = "/content/train_identity.csv"

In [ ]:
# =========================
# IEEE Fraud Detection v2
# SMOTE + Undersampling | LightGBM + XGBoost + CatBoost | Optuna
# =========================

import os, time, warnings, pickle, gc
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, classification_report, confusion_matrix
)

from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

warnings.filterwarnings("ignore")

import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# =========================
# CONFIG
# =========================
RANDOM_STATE      = 42
TRAIN_TRANSACTION = "/content/train_transaction.csv"
TRAIN_IDENTITY    = "/content/train_identity.csv"
USE_IDENTITY      = True
TARGET_COL        = "isFraud"
ID_COL            = "TransactionID"
N_OPTUNA_TRIALS   = 30    # زيد لـ 50-100 للنتائج أفضل
CV_FOLDS          = 3
SAVE_PATH         = "best_models_v2.pkl"

# اختار استراتيجية الـ Resampling:
# "smote_tomek"  ← SMOTE + Tomek Links (موصى به)
# "smote_enn"    ← SMOTE + ENN (أكثر تنظيفاً لكن أبطأ)
# "smote_under"  ← SMOTE + RandomUnderSampler (أسرع)
RESAMPLE_STRATEGY = "smote_tomek"

# نسبة الـ Resampling — None = تلقائي
# مثال: {"minority": 0.3, "majority": 1.0}
# يعني ارفع الفرود لـ 30% من الأكثرية، ثم نزّل الأكثرية للنسبة الطبيعية
SMOTE_RATIO      = 0.3   # fraud / normal بعد SMOTE
UNDER_RATIO      = 1.0   # استخدم كل الفرود بعد SMOTE مع تقليل النورمال

# =========================
# MEMORY REDUCTION
# =========================
def reduce_mem(df):
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = df[col].astype(np.float32)
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = df[col].astype(np.int32)
    return df

# =========================
# LOAD DATA
# =========================
print("📂 تحميل البيانات...")
df = pd.read_csv(TRAIN_TRANSACTION)
df = reduce_mem(df)

if USE_IDENTITY:
    df_id = pd.read_csv(TRAIN_IDENTITY)
    df_id = reduce_mem(df_id)
    df = df.merge(df_id, on=ID_COL, how="left")
    del df_id
    gc.collect()

print(f"   Shape: {df.shape}")

# =========================
# FEATURE ENGINEERING
# =========================
print("⚙️  Feature Engineering...")
df["hour"] = ((df["TransactionDT"] // 3600) % 24).astype(np.int8)
df["day"]  = ((df["TransactionDT"] // (3600*24)) % 7).astype(np.int8)
df.drop(columns=["TransactionDT"], inplace=True)
df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

# ميزات إضافية تساعد في الفرود
df["TransactionAmt_decimal"] = (df["TransactionAmt"] % 1).round(3)  # الكسور المشبوهة
df["TransactionAmt_rounded"] = (df["TransactionAmt"] % 1 == 0).astype(int)  # مبالغ مقربة

if "P_emaildomain" in df.columns:
    df["P_is_free_email"] = df["P_emaildomain"].str.contains(
        "gmail|yahoo|hotmail", na=False).astype(int)

if "R_emaildomain" in df.columns:
    df["R_is_free_email"] = df["R_emaildomain"].str.contains(
        "gmail|yahoo|hotmail", na=False).astype(int)
    df["same_email_domain"] = (df["P_emaildomain"] == df["R_emaildomain"]).astype(int)

# =========================
# PREP
# =========================
y = df[TARGET_COL]
df.drop(columns=[TARGET_COL, ID_COL], inplace=True)

fraud_rate = y.mean()
print(f"   نسبة الفرود في البيانات الأصلية: {fraud_rate:.2%}  ({y.sum():,} فرود / {len(y):,} إجمالي)")

num_cols = df.select_dtypes(include=["number"]).columns.tolist()
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

# =========================
# SPLIT  (قبل الـ Resampling عشان ما يصير data leakage)
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    df, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
del df
gc.collect()

# =========================
# PREPROCESS
# =========================
num_pipe = SkPipeline([("imp", SimpleImputer(strategy="median"))])
cat_pipe = SkPipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

print("🔧 تجهيز البيانات...")
preprocess.fit(X_train)
X_train_proc = preprocess.transform(X_train)
X_test_proc  = preprocess.transform(X_test)

# =========================
# RESAMPLING  (على train فقط)
# =========================
print(f"\n⚖️  Resampling بـ {RESAMPLE_STRATEGY}...")
print(f"   قبل: {np.bincount(y_train)}")

if RESAMPLE_STRATEGY == "smote_tomek":
    smote    = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=RANDOM_STATE, k_neighbors=5)
    resampler = SMOTETomek(smote=smote, random_state=RANDOM_STATE)

elif RESAMPLE_STRATEGY == "smote_enn":
    smote    = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=RANDOM_STATE, k_neighbors=5)
    resampler = SMOTEENN(smote=smote, random_state=RANDOM_STATE)

elif RESAMPLE_STRATEGY == "smote_under":
    smote = SMOTE(sampling_strategy=SMOTE_RATIO, random_state=RANDOM_STATE, k_neighbors=5)
    X_temp, y_temp = smote.fit_resample(X_train_proc, y_train)
    under = RandomUnderSampler(sampling_strategy=UNDER_RATIO, random_state=RANDOM_STATE)
    X_train_res, y_train_res = under.fit_resample(X_temp, y_temp)
    resampler = None

if RESAMPLE_STRATEGY != "smote_under":
    X_train_res, y_train_res = resampler.fit_resample(X_train_proc, y_train)

y_train_res = pd.Series(y_train_res)
print(f"   بعد:  {np.bincount(y_train_res)}")
new_fraud_rate = y_train_res.mean()
print(f"   نسبة الفرود بعد الـ Resampling: {new_fraud_rate:.2%}")

# scale_pos_weight بعد الـ Resampling
scale_pos = (y_train_res == 0).sum() / (y_train_res == 1).sum()

# =========================
# THRESHOLD
# =========================
def best_threshold(y_true, y_proba):
    p, r, thr = precision_recall_curve(y_true, y_proba)
    f1 = 2*p*r/(p+r+1e-9)
    return thr[np.argmax(f1[:-1])]

# =========================
# CROSS-VALIDATION HELPER
# =========================
def cv_score(model_fn, X, y, n_splits=CV_FOLDS):
    """CV على البيانات بعد الـ Resampling"""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    X = np.array(X)
    y = np.array(y)
    for tr_idx, val_idx in skf.split(X, y):
        X_tr,  X_val  = X[tr_idx],  X[val_idx]
        y_tr,  y_val  = y[tr_idx],  y[val_idx]
        model = model_fn()
        model.fit(X_tr, y_tr)
        probs = model.predict_proba(X_val)[:, 1]
        scores.append(average_precision_score(y_val, probs))
    return np.mean(scores)

# =============================
# OPTUNA — LightGBM
# =============================
print(f"\n🔍 Optuna — LightGBM ({N_OPTUNA_TRIALS} trials)...")

def lgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "num_leaves":        trial.suggest_int("num_leaves", 20, 300),
        "max_depth":         trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "feature_fraction":  trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction":  trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq":      trial.suggest_int("bagging_freq", 1, 7),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        # بعد SMOTE ما نحتاج scale_pos_weight لأن البيانات أكثر توازن
        "random_state":      RANDOM_STATE,
        "n_jobs":            -1,
        "verbosity":         -1,
    }
    return cv_score(lambda: lgb.LGBMClassifier(**params), X_train_res, y_train_res)

study_lgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_lgb.optimize(lgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
best_params_lgb = {**study_lgb.best_params, "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": -1}
print(f"✅ LightGBM — أفضل PR-AUC في CV: {study_lgb.best_value:.4f}")

# =============================
# OPTUNA — XGBoost
# =============================
print(f"\n🔍 Optuna — XGBoost ({N_OPTUNA_TRIALS} trials)...")

def xgb_objective(trial):
    params = {
        "n_estimators":      trial.suggest_int("n_estimators", 200, 1500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "max_depth":         trial.suggest_int("max_depth", 3, 10),
        "min_child_weight":  trial.suggest_int("min_child_weight", 1, 50),
        "subsample":         trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree":  trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 1.0),
        "reg_alpha":         trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda":        trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "gamma":             trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "eval_metric":       "aucpr",
        "random_state":      RANDOM_STATE,
        "n_jobs":            -1,
        "verbosity":         0,
    }
    return cv_score(lambda: XGBClassifier(**params), X_train_res, y_train_res)

study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_xgb.optimize(xgb_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
best_params_xgb = {**study_xgb.best_params, "eval_metric": "aucpr",
                   "random_state": RANDOM_STATE, "n_jobs": -1, "verbosity": 0}
print(f"✅ XGBoost — أفضل PR-AUC في CV: {study_xgb.best_value:.4f}")

# =============================
# OPTUNA — CatBoost
# =============================
print(f"\n🔍 Optuna — CatBoost ({N_OPTUNA_TRIALS} trials)...")

def cat_objective(trial):
    params = {
        "iterations":        trial.suggest_int("iterations", 200, 1500),
        "learning_rate":     trial.suggest_float("learning_rate", 0.005, 0.2, log=True),
        "depth":             trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg":       trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "random_strength":   trial.suggest_float("random_strength", 1e-9, 10.0, log=True),
        "border_count":      trial.suggest_int("border_count", 32, 255),
        "eval_metric":       "PRAUC",
        "random_seed":       RANDOM_STATE,
        "verbose":           False,
        "thread_count":      -1,
    }
    return cv_score(lambda: CatBoostClassifier(**params), X_train_res, y_train_res)

study_cat = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
study_cat.optimize(cat_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
best_params_cat = {**study_cat.best_params, "eval_metric": "PRAUC",
                   "random_seed": RANDOM_STATE, "verbose": False, "thread_count": -1}
print(f"✅ CatBoost — أفضل PR-AUC في CV: {study_cat.best_value:.4f}")

# =========================
# TRAIN FINAL MODELS
# =========================
print("\n▶ تدريب النماذج النهائية على كامل بيانات Train المعالجة...")

final_lgb = lgb.LGBMClassifier(**best_params_lgb)
final_lgb.fit(X_train_res, y_train_res)

final_xgb = XGBClassifier(**best_params_xgb)
final_xgb.fit(X_train_res, y_train_res)

final_cat = CatBoostClassifier(**best_params_cat)
final_cat.fit(X_train_res, y_train_res)

# =========================
# EVALUATE  (على Test الأصلي بدون Resampling)
# =========================
print("\n" + "="*55)
print("📊 النتائج على بيانات Test الأصلية (بدون Resampling)")
print("="*55)

results = {}
for name, model in [("LightGBM", final_lgb), ("XGBoost", final_xgb), ("CatBoost", final_cat)]:
    probs_test  = model.predict_proba(X_test_proc)[:, 1]
    probs_train = model.predict_proba(X_train_res)[:, 1]

    # Threshold محسّن من Train الـ Resampled
    thr   = best_threshold(y_train_res, probs_train)
    preds = (probs_test >= thr).astype(int)

    pr  = average_precision_score(y_test, probs_test)
    roc = roc_auc_score(y_test, probs_test)
    cm  = confusion_matrix(y_test, preds)

    results[name] = {"PR_AUC": pr, "ROC_AUC": roc, "threshold": thr,
                     "model": model, "confusion_matrix": cm}

    tn, fp, fn, tp = cm.ravel()
    print(f"\n📌 {name}:")
    print(f"   PR-AUC   : {pr:.4f}")
    print(f"   ROC-AUC  : {roc:.4f}")
    print(f"   Threshold: {thr:.4f}")
    print(f"   Confusion Matrix: TP={tp} | FP={fp} | TN={tn} | FN={fn}")
    print(classification_report(y_test, preds, digits=4))

# =========================
# ENSEMBLE  (متوسط الاحتمالات)
# =========================
print("="*55)
print("🤝 Ensemble (متوسط الاحتمالات للنماذج الثلاثة)")
print("="*55)

ens_probs = np.mean([
    final_lgb.predict_proba(X_test_proc)[:, 1],
    final_xgb.predict_proba(X_test_proc)[:, 1],
    final_cat.predict_proba(X_test_proc)[:, 1],
], axis=0)

# threshold من train probs المتوسطة
ens_train_probs = np.mean([
    final_lgb.predict_proba(X_train_res)[:, 1],
    final_xgb.predict_proba(X_train_res)[:, 1],
    final_cat.predict_proba(X_train_res)[:, 1],
], axis=0)
ens_thr   = best_threshold(y_train_res, ens_train_probs)
ens_preds = (ens_probs >= ens_thr).astype(int)

ens_pr  = average_precision_score(y_test, ens_probs)
ens_roc = roc_auc_score(y_test, ens_probs)
ens_cm  = confusion_matrix(y_test, ens_preds)
tn, fp, fn, tp = ens_cm.ravel()

results["Ensemble"] = {"PR_AUC": ens_pr, "ROC_AUC": ens_roc,
                        "threshold": ens_thr, "confusion_matrix": ens_cm}

print(f"\n📌 Ensemble:")
print(f"   PR-AUC   : {ens_pr:.4f}")
print(f"   ROC-AUC  : {ens_roc:.4f}")
print(f"   Threshold: {ens_thr:.4f}")
print(f"   Confusion Matrix: TP={tp} | FP={fp} | TN={tn} | FN={fn}")
print(classification_report(y_test, ens_preds, digits=4))

# =========================
# اختيار الأفضل
# =========================
best_name = max(
    [k for k in results if k != "Ensemble"],
    key=lambda k: results[k]["PR_AUC"]
)
print("="*55)
print(f"🏆 أفضل موديل منفرد: {best_name} — PR-AUC = {results[best_name]['PR_AUC']:.4f}")
print(f"🤝 Ensemble           — PR-AUC = {results['Ensemble']['PR_AUC']:.4f}")

# مقارنة مع النتائج القديمة
print("\n📈 مقارنة مع النسخة السابقة (بدون SMOTE):")
print(f"   LightGBM : 0.5153 → {results['LightGBM']['PR_AUC']:.4f}")
print(f"   XGBoost  : 0.5351 → {results['XGBoost']['PR_AUC']:.4f}")
print(f"   CatBoost : جديد  → {results['CatBoost']['PR_AUC']:.4f}")
print(f"   Ensemble : جديد  → {results['Ensemble']['PR_AUC']:.4f}")

# =========================
# SAVE
# =========================
with open(SAVE_PATH, "wb") as f:
    pickle.dump({
        "preprocessor":     preprocess,
        "resample_strategy": RESAMPLE_STRATEGY,
        "lgb_model":        final_lgb,
        "xgb_model":        final_xgb,
        "cat_model":        final_cat,
        "lgb_best_params":  best_params_lgb,
        "xgb_best_params":  best_params_xgb,
        "cat_best_params":  best_params_cat,
        "lgb_study":        study_lgb,
        "xgb_study":        study_xgb,
        "cat_study":        study_cat,
        "results":          {k: {kk: vv for kk, vv in v.items() if kk != "model"}
                             for k, v in results.items()},
        "best_model_name":  best_name,
    }, f)

print(f"\n💾 تم الحفظ: {SAVE_PATH}")

📂 تحميل البيانات...
   Shape: (590540, 434)
⚙️  Feature Engineering...
   نسبة الفرود في البيانات الأصلية: 3.50%  (20,663 فرود / 590,540 إجمالي)
🔧 تجهيز البيانات...

⚖️  Resampling بـ smote_tomek...
   قبل: [455902  16530]
   بعد:  [454647 135515]
   نسبة الفرود بعد الـ Resampling: 22.96%

🔍 Optuna — LightGBM (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

✅ LightGBM — أفضل PR-AUC في CV: 0.9937

🔍 Optuna — XGBoost (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

✅ XGBoost — أفضل PR-AUC في CV: 0.9918

🔍 Optuna — CatBoost (30 trials)...


  0%|          | 0/30 [00:00<?, ?it/s]

Training has stopped (degenerate solution on iteration 1316, probably too small l2-regularization, try to increase it)
Training has stopped (degenerate solution on iteration 419, probably too small l2-regularization, try to increase it)


✅ CatBoost — أفضل PR-AUC في CV: 0.9915

▶ تدريب النماذج النهائية على كامل بيانات Train المعالجة...

📊 النتائج على بيانات Test الأصلية (بدون Resampling)

📌 LightGBM:
   PR-AUC   : 0.8790
   ROC-AUC  : 0.9731
   Threshold: 0.8229
   Confusion Matrix: TP=2608 | FP=43 | TN=113932 | FN=1525
              precision    recall  f1-score   support

           0     0.9868    0.9996    0.9932    113975
           1     0.9838    0.6310    0.7689      4133

    accuracy                         0.9867    118108
   macro avg     0.9853    0.8153    0.8810    118108
weighted avg     0.9867    0.9867    0.9853    118108


📌 XGBoost:
   PR-AUC   : 0.8114
   ROC-AUC  : 0.9642
   Threshold: 0.2758
   Confusion Matrix: TP=2845 | FP=446 | TN=113529 | FN=1288
              precision    recall  f1-score   support

           0     0.9888    0.9961    0.9924    113975
           1     0.8645    0.6884    0.7664      4133

    accuracy                         0.9853    118108
   macro avg     0.9266    0.8422